In [1]:
import pandas as pd
import numpy as np

# === Load data ===
yt_x_survey = pd.read_csv("../../../data/raw_yt_x_survey.csv")
ts_tt_survey = pd.read_csv("../../../data/raw_tt_ts_survey.csv")
pairs_yt_x = pd.read_csv("../../../data/pairs_data_yt_x.csv")
pairs_ts_tt = pd.read_csv("../../../data/pairs_data_tt_ts.csv")

# Clean column names
for df in [yt_x_survey, ts_tt_survey, pairs_yt_x, pairs_ts_tt]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if "id" in col.lower():
            df[col] = df[col].astype(str).str.strip()

# Filter to participants who completed the task
yt_x_filtered = yt_x_survey[yt_x_survey["response_id"].isin(pairs_yt_x["response_id"])]
ts_tt_filtered = ts_tt_survey[ts_tt_survey["response_id"].isin(pairs_ts_tt["response_id"])]

print(f"YouTube/X participants: {len(yt_x_filtered)}")
print(f"TikTok/Truth Social participants: {len(ts_tt_filtered)}")

# === Map item numbers to source names ===
newstrust_items = {
    "newstrust_1": "Newspapers",
    "newstrust_2": "Social media",
    "newstrust_3": "News websites",
    "newstrust_4": "Radio news",
    "newstrust_5": "Local news organizations",
    "newstrust_6": "National news organizations"
}

def get_response_counts(df, label):
    """Get counts and percentages for each response option"""
    rows = []
    total = len(df)
    
    # Politics
    if "politics" in df.columns:
        pol_counts = df["politics"].value_counts(dropna=False).sort_index()
        for response, count in pol_counts.items():
            pct = round(count / total * 100, 1)
            rows.append({
                "Measure": "Interest in Politics",
                "Response": response,
                "Count": count,
                "Percent": pct
            })
    
    # Each newstrust item
    for col, source in newstrust_items.items():
        if col in df.columns:
            trust_counts = df[col].value_counts(dropna=False).sort_index()
            for response, count in trust_counts.items():
                pct = round(count / total * 100, 1)
                rows.append({
                    "Measure": f"Trust - {source}",
                    "Response": response,
                    "Count": count,
                    "Percent": pct
                })
    
    summary_df = pd.DataFrame(rows)
    out_name = f"../../../output/survey_response_counts_{label.lower().replace('/', '_')}.csv"
    summary_df.to_csv(out_name, index=False)
    print(f"\nSaved → {out_name}")
    return summary_df

# === Generate count summaries ===
print("\n=== YouTube/X Participants ===")
yt_x_counts = get_response_counts(yt_x_filtered, "YouTube_X")
print(yt_x_counts.to_string(index=False))

print("\n=== TikTok/Truth Social Participants ===")
ts_tt_counts = get_response_counts(ts_tt_filtered, "TikTok_TruthSocial")
print(ts_tt_counts.to_string(index=False))

# === Optional: Combined table ===
yt_x_counts["Group"] = "YouTube/X"
ts_tt_counts["Group"] = "TikTok/Truth Social"

combined = pd.concat([yt_x_counts, ts_tt_counts])
combined = combined[["Group", "Measure", "Response", "Count", "Percent"]]
combined.to_csv("../../../output/survey_response_counts_combined.csv", index=False)
print("\n=== Combined Response Counts ===")
print(combined.to_string(index=False))

YouTube/X participants: 601
TikTok/Truth Social participants: 606

=== YouTube/X Participants ===

Saved → ../../../output/survey_response_counts_youtube_x.csv
                            Measure           Response  Count  Percent
               Interest in Politics  Extremely closely    106     17.6
               Interest in Politics Moderately closely    204     33.9
               Interest in Politics         Not at all     15      2.5
               Interest in Politics   Slightly closely     69     11.5
               Interest in Politics       Very closely    207     34.4
                 Trust - Newspapers           A little    157     26.1
                 Trust - Newspapers       All the time     16      2.7
                 Trust - Newspapers         Not at all     98     16.3
                 Trust - Newspapers        Quite a lot    151     25.1
                 Trust - Newspapers               Some    179     29.8
               Trust - Social media           A little    1

In [2]:
import pandas as pd

# Load data
yt_x_survey = pd.read_csv("../../../data/raw_yt_x_survey.csv")
ts_tt_survey = pd.read_csv("../../../data/raw_tt_ts_survey.csv")
pairs_yt_x = pd.read_csv("../../../data/pairs_data_yt_x.csv")
pairs_ts_tt = pd.read_csv("../../../data/pairs_data_tt_ts.csv")

# Clean
for df in [yt_x_survey, ts_tt_survey, pairs_yt_x, pairs_ts_tt]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if "id" in col.lower():
            df[col] = df[col].astype(str).str.strip()

# Filter
yt_x_filtered = yt_x_survey[yt_x_survey["response_id"].isin(pairs_yt_x["response_id"])]
ts_tt_filtered = ts_tt_survey[ts_tt_survey["response_id"].isin(pairs_ts_tt["response_id"])]

# Cynicism items
cynicism_items = {
    "sm_cynicism_1": "You cannot trust the news stories people share on social media",
    "sm_cynicism_2": "Too often credible news information is mixed up with misinformation on social media",
    "sm_cynicism_3": "You should not rely on social media for news information",
    "sm_cynicism_4": "I feel confident that I can find the truth about political issues"
}

def get_cynicism_counts(df, label):
    """Get counts for each cynicism item"""
    rows = []
    total = len(df)
    
    for col, item_text in cynicism_items.items():
        if col in df.columns:
            counts = df[col].value_counts(dropna=False).sort_index()
            for response, count in counts.items():
                pct = round(count / total * 100, 1)
                rows.append({
                    "Item": item_text,
                    "Response": response,
                    "Count": count,
                    "Percent": pct
                })
    
    summary_df = pd.DataFrame(rows)
    out_name = f"../../../output/cynicism_counts_{label.lower().replace('/', '_')}.csv"
    summary_df.to_csv(out_name, index=False)
    print(f"Saved → {out_name}")
    return summary_df

print("=== YouTube/X ===")
yt_x_cyn = get_cynicism_counts(yt_x_filtered, "YouTube_X")
print(yt_x_cyn.to_string(index=False))

print("\n=== TikTok/Truth Social ===")
ts_tt_cyn = get_cynicism_counts(ts_tt_filtered, "TikTok_TruthSocial")
print(ts_tt_cyn.to_string(index=False))

=== YouTube/X ===
Saved → ../../../output/cynicism_counts_youtube_x.csv
                                                                               Item          Response  Count  Percent
                     You cannot trust the news stories people share on social media             Agree    150     25.0
                     You cannot trust the news stories people share on social media          Disagree     13      2.2
                     You cannot trust the news stories people share on social media           Neutral     90     15.0
                     You cannot trust the news stories people share on social media    Somewhat Agree    193     32.1
                     You cannot trust the news stories people share on social media Somewhat Disagree     34      5.7
                     You cannot trust the news stories people share on social media    Strongly Agree    111     18.5
                     You cannot trust the news stories people share on social media Strongly Disagree 

In [3]:
import pandas as pd

# Load data
yt_x_survey = pd.read_csv("../../../data/raw_yt_x_survey.csv")
ts_tt_survey = pd.read_csv("../../../data/raw_tt_ts_survey.csv")
pairs_yt_x = pd.read_csv("../../../data/pairs_data_yt_x.csv")
pairs_ts_tt = pd.read_csv("../../../data/pairs_data_tt_ts.csv")

# Clean
for df in [yt_x_survey, ts_tt_survey, pairs_yt_x, pairs_ts_tt]:
    df.columns = df.columns.str.strip()
    for col in df.columns:
        if "id" in col.lower():
            df[col] = df[col].astype(str).str.strip()

# Filter
yt_x_filtered = yt_x_survey[yt_x_survey["response_id"].isin(pairs_yt_x["response_id"])]
ts_tt_filtered = ts_tt_survey[ts_tt_survey["response_id"].isin(pairs_ts_tt["response_id"])]

# Conspiracy items
conspiracy_items = {
    "conspiracies_1": "Much of our lives are being controlled by plots hatched in secret places",
    "conspiracies_2": "Even though we live in a democracy a few people will always run things anyway",
    "conspiracies_3": "The people who really 'run' the country are not known to the voter",
    "conspiracies_4": "Big events like wars, recessions, and the outcomes of elections are controlled by small groups of people who are working in secret against the rest of us"
}

def get_conspiracy_counts(df, label):
    """Get counts for each conspiracy item"""
    rows = []
    total = len(df)
    
    for col, item_text in conspiracy_items.items():
        if col in df.columns:
            counts = df[col].value_counts(dropna=False).sort_index()
            for response, count in counts.items():
                pct = round(count / total * 100, 1)
                rows.append({
                    "Item": item_text,
                    "Response": response,
                    "Count": count,
                    "Percent": pct
                })
    
    summary_df = pd.DataFrame(rows)
    out_name = f"../../../output/conspiracy_counts_{label.lower().replace('/', '_')}.csv"
    summary_df.to_csv(out_name, index=False)
    print(f"Saved → {out_name}")
    return summary_df

print("=== YouTube/X ===")
yt_x_con = get_conspiracy_counts(yt_x_filtered, "YouTube_X")

print("\n=== TikTok/Truth Social ===")
ts_tt_con = get_conspiracy_counts(ts_tt_filtered, "TikTok_TruthSocial")

=== YouTube/X ===
Saved → ../../../output/conspiracy_counts_youtube_x.csv

=== TikTok/Truth Social ===
Saved → ../../../output/conspiracy_counts_tiktok_truthsocial.csv
